# Reconciliation of two DP-FedAvg sweeps

Compute forecasts and metrics for **base, mints, wlsv, and ols** from the 12 completed 20-round sweep runs for each of GEF2017 and AusGrid2013.

- Reuse the original implementation in `shared_modules/FHF_Cloud.py`: mints = `mint_shrinkage`, wlsv = `mint_var`, ols = `mint_ols`, and `ridge=1e-6`.
- Estimate matrices from the **training residuals** saved for each run. Test observations are used only for scoring. No retraining or additional forecast or residual noise is applied.
- MASE / RMSSE use the same lag-1 training-target difference denominators as the original sweep. Compute metrics per series, then take the arithmetic mean across all series.
- Recompute on every execution and verify agreement with the original base metrics, `P S ≈ S`, `P² ≈ P`, and hierarchical aggregation consistency of reconciled forecasts.
- Save all results to **`DP_fhf/outputs/dp_sweep_reconciliation/`**, preserving the input sweeps.

`training_epsilon` / `training_delta` record only the original DP-FedAvg training privacy parameters. This training-residual covariance estimate has no additional DP guarantee.

Directory migration: existing computed results were moved to the directory above, and the result tables below were retained. Logged file paths were updated to their current locations. Future runs also write to this directory.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

# Start from any project subdirectory, or specify the project root with an environment variable.
candidates = []
if os.environ.get("DP_FHF_PROJECT_ROOT"):
    candidates.append(Path(os.environ["DP_FHF_PROJECT_ROOT"]).expanduser())
candidates.extend([Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "PPHF_modules" / "fhf").is_dir() and (p / "DP_fhf").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DP_FHF_PROJECT_ROOT to the FHF_NDSH project directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PPHF_modules.dp_fhf import sweep_reconciliation
importlib.reload(sweep_reconciliation)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
DATASETS = ("GEF2017", "AusGrid2013")
OUTPUT_ROOT = ROOT / "DP_fhf" / "outputs" / "dp_sweep_reconciliation"
RIDGE = 1e-6
SAVE_FORECASTS = True  # Save pointwise forecasts as CSV.gz files, readable directly with pandas.read_csv.
RUN_IDS = None         # None = all 24 runs; use ["nm_0p3__clip_1"] to check a subset.

print("Output:", OUTPUT_ROOT)
print("Methods:", sweep_reconciliation.METHODS)
print("Matrix source: each run's saved training residuals")

## Compute and save

The larger AusGrid outputs take some time to compress and write. Display MASE for all four methods after each run completes.
Recompute and replace existing results with the same names generated by this notebook. Publish a run only after all its checks and file writes succeed.

In [ ]:
summary = sweep_reconciliation.run_sweep_reconciliation(
    root=ROOT,
    datasets=DATASETS,
    output_root=OUTPUT_ROOT,
    ridge=RIDGE,
    save_forecasts=SAVE_FORECASTS,
    run_ids=RUN_IDS,
)
assert len(summary) == 4 * summary[["dataset", "run_id"]].drop_duplicates().shape[0]
print("Completed method rows:", len(summary))

## MASE / RMSSE for each parameter combination

Sort by clipping and noise to compare changes in noise. The base is the original forecast from the same run.

In [ ]:
for dataset in DATASETS:
    print(f"\n{dataset}")
    table = summary.loc[summary.dataset == dataset].pivot(
        index=["clip_norm", "noise_multiplier"], columns="method", values=["MASE", "RMSSE"]
    ).sort_index()
    table = table.reindex(columns=pd.MultiIndex.from_product(
        [["MASE", "RMSSE"], ["base", "mints", "wlsv", "ols"]], names=table.columns.names))
    display(table)

## Changes relative to the matching base

Negative values indicate lower error. Each dataset / method has 12 parameter combinations. These are descriptive comparisons, not statistical conclusions across multiple seeds.

In [ ]:
comparison_columns = ["dataset", "run_id", "method", "MASE", "RMSSE",
                      "change_MASE_pct_vs_base", "change_RMSSE_pct_vs_base"]
display(summary.loc[summary.method != "base", comparison_columns])
best = summary.loc[summary.groupby(["dataset", "method"])["MASE"].idxmin(),
                   ["dataset", "method", "run_id", "clip_norm", "noise_multiplier", "MASE", "RMSSE"]]
print("Lowest mean MASE within each dataset / method:")
display(best.sort_values(["dataset", "method"]))

## Validation and file locations

- `metrics_summary.csv`: all 96 rows (24 runs × base + 3 reconciliation methods), including four metrics and changes relative to base.
- `metrics_wide.csv`: 24 rows with metrics for the four methods side by side.
- `per_node_metrics.csv`: all per-series metrics.
- `matrix_diagnostics.csv`: hierarchy consistency and matrix validation.
- `<dataset>/<run_id>/`: run metrics, `reconciliation_matrices.npz` (S / G / P / W), `reconciled_forecasts.csv.gz`, and input SHA256 hashes / computation settings.
- `manifest.json`: completion record for the entire batch. An interrupted execution does not retain an old full-batch success marker.

In [ ]:
import json
manifest = json.loads((OUTPUT_ROOT / "manifest.json").read_text())
checks = pd.read_csv(OUTPUT_ROOT / "matrix_diagnostics.csv")
assert manifest["status"] == "complete"
assert checks["PS_minus_S_max_abs"].max() < 1e-7
assert checks["coherence_relative"].max() < 1e-9
print("Runs:", manifest["runs_completed"])
print("Maximum |PS-S|:", checks["PS_minus_S_max_abs"].max())
print("Maximum relative coherence error:", checks["coherence_relative"].max())
print("Summary:", OUTPUT_ROOT / "metrics_summary.csv")
print("Wide comparison:", OUTPUT_ROOT / "metrics_wide.csv")
# Example: pd.read_csv(OUTPUT_ROOT / "GEF2017" / "nm_0p3__clip_1" / "reconciled_forecasts.csv.gz")